# Graph Basics for Knowledge Graphs

Graphs are mathematical structures used to model **relationships among objects**. In agentic AI systems, knowledge graphs serve as a **semantic memory** — a persistent, queryable structure that captures entities, relationships, and context over time.

This notebook covers:
- Directed, undirected, and weighted graphs
- Building and visualizing graphs with NetworkX
- Graph algorithms relevant to agentic AI (centrality, PageRank, shortest paths)

In [ ]:
# %pip install networkx matplotlib -q

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

## 1. Directed Graphs

In a **directed graph**, edges have direction (A → B). This is the most common form in knowledge graphs, where relationships are asymmetric: *Alice works_at Google* does not imply *Google works_at Alice*.

**Example:** From the sentence *"Alan Turing worked at Bletchley Park during WWII"*, we extract:
- **Vertices**: `Alan Turing`, `Bletchley Park`, `WWII`
- **Edges**: `(Alan Turing) —worked_at→ (Bletchley Park)`, `(Alan Turing) —active_during→ (WWII)`

In [ ]:
G_directed = nx.DiGraph()

# Add edges with relationship labels
G_directed.add_edge("Alan Turing", "Bletchley Park", relation="worked_at")
G_directed.add_edge("Alan Turing", "WWII", relation="active_during")

plt.figure(figsize=(8, 5))
pos = nx.spring_layout(G_directed, seed=42)
nx.draw(G_directed, pos, with_labels=True, node_color="lightblue",
        node_size=2500, font_size=11, font_weight="bold",
        arrows=True, arrowsize=20, edge_color="gray")

edge_labels = nx.get_edge_attributes(G_directed, "relation")
nx.draw_networkx_edge_labels(G_directed, pos, edge_labels=edge_labels, font_size=10)

plt.title("Directed Graph: Alan Turing Knowledge")
plt.tight_layout()
plt.show()

print(f"Nodes: {list(G_directed.nodes())}")
print(f"Edges: {[(u, v, d['relation']) for u, v, d in G_directed.edges(data=True)]}")

## 2. Undirected Graphs

In an **undirected graph**, relationships are symmetrical (A—B). This is useful for modeling mutual relationships like *married_to* or *collaborates_with*.

In [ ]:
G_undirected = nx.Graph()

G_undirected.add_edge("Alice", "Bob", relation="married_to")
G_undirected.add_edge("Alice", "Carol", relation="friends_with")
G_undirected.add_edge("Bob", "Dave", relation="colleagues")
G_undirected.add_edge("Carol", "Dave", relation="friends_with")

plt.figure(figsize=(8, 5))
pos = nx.spring_layout(G_undirected, seed=42)
nx.draw(G_undirected, pos, with_labels=True, node_color="lightgreen",
        node_size=2500, font_size=11, font_weight="bold", edge_color="gray")

edge_labels = nx.get_edge_attributes(G_undirected, "relation")
nx.draw_networkx_edge_labels(G_undirected, pos, edge_labels=edge_labels, font_size=10)

plt.title("Undirected Graph: Social Relationships")
plt.tight_layout()
plt.show()

## 3. Weighted Graphs

In a **weighted graph**, edges have numerical values indicating strength, confidence, or cost. In agentic systems, weights can represent an agent's confidence in a capability or the quality score of a task assignment.

In [ ]:
G_weighted = nx.DiGraph()

# Agent-task assignments with confidence scores
G_weighted.add_edge("ResearchAgent", "Summarize Paper", weight=0.95, relation="capable_of")
G_weighted.add_edge("ResearchAgent", "Web Search", weight=0.80, relation="capable_of")
G_weighted.add_edge("CodeAgent", "Write Code", weight=0.92, relation="capable_of")
G_weighted.add_edge("CodeAgent", "Debug Code", weight=0.88, relation="capable_of")
G_weighted.add_edge("CodeAgent", "Summarize Paper", weight=0.45, relation="capable_of")

plt.figure(figsize=(9, 6))
pos = nx.spring_layout(G_weighted, seed=42)

# Edge width proportional to weight
weights = [G_weighted[u][v]["weight"] * 3 for u, v in G_weighted.edges()]
nx.draw(G_weighted, pos, with_labels=True, node_color="lightyellow",
        node_size=3000, font_size=10, font_weight="bold",
        arrows=True, arrowsize=15, width=weights, edge_color="steelblue")

edge_labels = {(u, v): f"{d['weight']:.2f}" for u, v, d in G_weighted.edges(data=True)}
nx.draw_networkx_edge_labels(G_weighted, pos, edge_labels=edge_labels, font_size=9)

plt.title("Weighted Graph: Agent Capabilities with Confidence Scores")
plt.tight_layout()
plt.show()

## 4. A More Complex Knowledge Graph: Agentic AI Scenario

Let's build a larger graph representing an agentic AI system with **agents**, **tasks**, **documents**, and **tools** connected by relationships like `ASSIGNED_TO`, `HAS_INPUT`, `USES_TOOL`, and `DEPENDS_ON`.

In [ ]:
import matplotlib.patches as mpatches

G = nx.DiGraph()

# Define node categories
agents = ["ResearchAgent", "SummaryAgent", "ValidationAgent"]
tasks = ["Task1:Research", "Task2:Summarize", "Task3:Validate"]
documents = ["Doc1:Paper", "Doc2:Dataset", "Doc3:Report"]
tools = ["WebSearch", "LLM", "Database"]

# Agent → Task assignments
G.add_edge("ResearchAgent", "Task1:Research", relation="ASSIGNED_TO")
G.add_edge("SummaryAgent", "Task2:Summarize", relation="ASSIGNED_TO")
G.add_edge("ValidationAgent", "Task3:Validate", relation="ASSIGNED_TO")

# Task → Document inputs
G.add_edge("Task1:Research", "Doc1:Paper", relation="HAS_INPUT")
G.add_edge("Task1:Research", "Doc2:Dataset", relation="HAS_INPUT")
G.add_edge("Task2:Summarize", "Doc1:Paper", relation="HAS_INPUT")
G.add_edge("Task3:Validate", "Doc3:Report", relation="HAS_INPUT")

# Agent → Tool usage
G.add_edge("ResearchAgent", "WebSearch", relation="USES_TOOL")
G.add_edge("ResearchAgent", "Database", relation="USES_TOOL")
G.add_edge("SummaryAgent", "LLM", relation="USES_TOOL")
G.add_edge("ValidationAgent", "LLM", relation="USES_TOOL")
G.add_edge("ValidationAgent", "Database", relation="USES_TOOL")

# Task dependencies
G.add_edge("Task2:Summarize", "Task1:Research", relation="DEPENDS_ON")
G.add_edge("Task3:Validate", "Task2:Summarize", relation="DEPENDS_ON")

# Color nodes by type
color_map = []
for node in G.nodes():
    if node in agents:
        color_map.append("#FF9999")
    elif node in tasks:
        color_map.append("#99CCFF")
    elif node in documents:
        color_map.append("#99FF99")
    else:
        color_map.append("#FFCC99")

plt.figure(figsize=(14, 9))
pos = nx.spring_layout(G, seed=42, k=2)
nx.draw(G, pos, with_labels=True, node_color=color_map,
        node_size=3000, font_size=9, font_weight="bold",
        arrows=True, arrowsize=15, edge_color="gray")

edge_labels = nx.get_edge_attributes(G, "relation")
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8)

legend_handles = [
    mpatches.Patch(color="#FF9999", label="Agents"),
    mpatches.Patch(color="#99CCFF", label="Tasks"),
    mpatches.Patch(color="#99FF99", label="Documents"),
    mpatches.Patch(color="#FFCC99", label="Tools"),
]
plt.legend(handles=legend_handles, loc="upper left", fontsize=11)
plt.title("Agentic AI Knowledge Graph", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Graph Algorithms for Agentic Systems

Graph algorithms help agents determine **relevance, influence, and routing**:

| Algorithm | Purpose | Agentic Use Case |
|-----------|---------|------------------|
| Degree centrality | How connected is each node? | Find most versatile agents |
| Betweenness centrality | Which nodes are critical connectors? | Identify bottlenecks |
| PageRank | Which nodes are most "important"? | Rank documents/agents by influence |
| Shortest path | How to route between entities? | Task routing and dependency chains |

In [ ]:
# Degree centrality
degree_cent = nx.degree_centrality(G)
print("=== Degree Centrality (most connected nodes) ===")
for node, score in sorted(degree_cent.items(), key=lambda x: x[1], reverse=True):
    print(f"  {node:25s} {score:.3f}")

In [ ]:
# Betweenness centrality: which nodes are critical connectors?
between_cent = nx.betweenness_centrality(G)
print("=== Betweenness Centrality (critical connectors) ===")
for node, score in sorted(between_cent.items(), key=lambda x: x[1], reverse=True)[:5]:
    print(f"  {node:25s} {score:.3f}")

In [ ]:
# PageRank: node importance based on link structure
pagerank = nx.pagerank(G)
print("=== PageRank (node importance) ===")
for node, score in sorted(pagerank.items(), key=lambda x: x[1], reverse=True):
    print(f"  {node:25s} {score:.3f}")

In [ ]:
# Shortest path: how does ResearchAgent connect to Doc3:Report?
source, target = "ResearchAgent", "Doc3:Report"
try:
    path = nx.shortest_path(G, source, target)
    print(f"Shortest path from {source} to {target}:")
    print("  " + " → ".join(path))
    print("\nEdges along the path:")
    for i in range(len(path) - 1):
        rel = G[path[i]][path[i+1]]["relation"]
        print(f"  ({path[i]}) —{rel}→ ({path[i+1]})")
except nx.NetworkXNoPath:
    print("No path found.")

In [ ]:
# Graph summary statistics
print("=== Graph Statistics ===")
print(f"  Nodes: {G.number_of_nodes()}")
print(f"  Edges: {G.number_of_edges()}")
print(f"  Density: {nx.density(G):.3f}")
print(f"  Is DAG (no cycles): {nx.is_directed_acyclic_graph(G)}")
print(f"  Weakly connected components: {nx.number_weakly_connected_components(G)}")

## Key Takeaways

- **Graphs make relationships explicit** — forming a foundation for reasoning, retrieval, and memory in agentic systems
- **Directed graphs** model asymmetric relationships (most common in KGs)
- **Weighted graphs** encode confidence or strength, useful for ranking and decision-making
- **Graph algorithms** (centrality, PageRank, shortest path) help agents reason about structure, find important entities, and route tasks efficiently
- In multi-agent systems, the knowledge graph serves as a **shared world model** enabling collaboration and grounding